<a href="https://colab.research.google.com/github/hemalathachereddy6-prog/NLP_INN226106902/blob/main/TASK_2(NLP)senti_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#step 1

import pandas as pd

# Load uploaded CSV files
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Check shapes
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Check columns
print("Columns in train dataset:", train_df.columns)

# Preview first 5 rows
train_df.head()

Train shape: (31962, 3)
Test shape: (17197, 2)
Columns in train dataset: Index(['id', 'label', 'tweet'], dtype='object')


,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [ ]:
train_df['label'].value_counts() #positive (1) vs negative (0) tweets

,count
label,
0,29720
1,2242


In [ ]:
#step 2

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TreebankWordTokenizer

# Download stopwords and wordnet (still needed)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
tokenizer = TreebankWordTokenizer()  # NEW tokenizer

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenize using TreebankWordTokenizer
    words = tokenizer.tokenize(text)

    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return ' '.join(words)

# Apply preprocessing
train_df['clean_tweet'] = train_df['tweet'].apply(preprocess_text)
test_df['clean_tweet'] = test_df['tweet'].apply(preprocess_text)

# Preview
train_df[['tweet','clean_tweet']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,tweet,clean_tweet
0,@user when a father is dysfunctional and is s...,father dysfunctional selfish drag kid dysfunction
1,@user @user thanks for #lyft credit i can't us...,thanks credit cant use cause dont offer wheelc...
2,bihday your majesty,bihday majesty
3,#model i love u take with u all the time in ...,love u take u time urð ðððð ððð
4,factsguide: society now #motivation,factsguide society


In [ ]:
#step 3

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)  # limit features to 5000 for speed

# Fit on train data and transform
X_train = tfidf.fit_transform(train_df['clean_tweet'])

# Transform test data
X_test = tfidf.transform(test_df['clean_tweet'])

# Labels
y_train = train_df['label']

print("Train features shape:", X_train.shape)
print("Test features shape:", X_test.shape)

Train features shape: (31962, 5000)
Test features shape: (17197, 5000)


In [ ]:
#step 4
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Dictionary to store models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# Dictionary to store results
results = {}

# Train each model and evaluate on training data
for name, model in models.items():
    model.fit(X_train, y_train)  # Train model
    y_pred = model.predict(X_train)  # Predict on training data

    # Evaluate
    results[name] = {
        "Accuracy": accuracy_score(y_train, y_pred),
        "Precision": precision_score(y_train, y_pred),
        "Recall": recall_score(y_train, y_pred),
        "F1 Score": f1_score(y_train, y_pred)
    }

# Convert results to DataFrame for easy comparison
results_df = pd.DataFrame(results).T
results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.949346,0.933241,0.299286,0.453225
Naive Bayes,0.948439,0.971429,0.272971,0.426184
Decision Tree,0.997122,0.998609,0.960303,0.979081


In [ ]:
# step 5
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# 1️.Split your training data into training and validation sets (80-20)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,       # 20% for validation
    random_state=42,     # reproducibility
    stratify=y_train     # keeps class distribution same in both sets
)

# 2️.Define your models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# 3️.Dictionary to store evaluation results
results = {}

# 4️.Train each model and evaluate on the validation set
for name, model in models.items():
    model.fit(X_train_split, y_train_split)   # Train on training split
    y_val_pred = model.predict(X_val)         # Predict on validation set

    # Calculate metrics
    results[name] = {
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1 Score": f1_score(y_val, y_val_pred)
    }

# 5️.Convert results to a DataFrame for easy comparison
results_df = pd.DataFrame(results).T
results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.944471,0.897436,0.234375,0.371681
Naive Bayes,0.945878,0.972222,0.234375,0.377698
Decision Tree,0.936180,0.548544,0.504464,0.525581




## **Step 6: Comparison & Insights**

### **1. Best Preprocessing Steps**

* **Lowercasing**: Standardizes text, reduces duplicate tokens.
* **Removing URLs, mentions (@user), and hashtags (#tag)**: Cleans noise specific to Twitter data.
* **Removing punctuation and special characters**: Keeps only meaningful words.
* **Stopwords removal**: Eliminates common words that do not contribute to sentiment.
* **Lemmatization**: Reduces words to their root form for better generalization (e.g., *running → run*).

These steps ensured that the models focus only on meaningful content, improving learning efficiency.


### **2. Best Vectorization**

* **TF-IDF (Term Frequency–Inverse Document Frequency)** was the most effective feature representation.

  * It assigns higher weight to important words and lowers the impact of frequent, uninformative words.
  * Outperformed Bag-of-Words in capturing meaningful distinctions in sentiment.


### **3. Best Model**

* **Depends on the metric of interest:**

  * **High accuracy & precision**: Logistic Regression / Naive Bayes
  * **Balanced detection of positive sentiment (better recall)**: Decision Tree

* In your results:

  * Logistic Regression: Accuracy ~94.4%, low recall
  * Naive Bayes: Accuracy ~94.6%, very high precision, low recall
  * Decision Tree: Accuracy ~93.6%, more balanced recall and F1 (~0.53)

**Recommendation:** For practical deployment, Logistic Regression or Naive Bayes is preferred if minimizing false positives is critical, while Decision Tree is better if detecting positive sentiment is prioritized.


### **4. Trade-offs**

| Model                             | Trade-offs                                                                                                |
| --------------------------------- | --------------------------------------------------------------------------------------------------------- |
| Logistic Regression / Naive Bayes | High precision, low recall → may miss positive tweets, good overall accuracy                              |
| Decision Tree                     | Balanced recall and precision → better for minority classes, slightly lower overall accuracy, may overfit |

